In [1]:
%pip install -q numpy==2.4.4 pandas==3.0.2 scikit-learn==1.8.0 joblib==1.5.3

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 02 - Quy trình xử lý dữ liệu

Mục tiêu của bước này là tạo dữ liệu đầu vào ổn định cho mọi model và cho AI Service.

**Nguyên tắc chống data leakage:** chia train/test trước; mọi imputer, encoder và scaler chỉ được `fit` trên `X_train`. Vì vậy toàn bộ tiền xử lý được đóng gói trong `ColumnTransformer`/`Pipeline`.

## Quyết định xử lý

| Bước | Quyết định | Lý do |
|---|---|---|
| Kiểm tra ban đầu | Kiểm tra kiểu dữ liệu, trùng lặp, missing và miền giá trị | Phát hiện dữ liệu không đáng tin trước khi train |
| Làm sạch | Bỏ `id`, bỏ dòng trùng chính xác; giữ ngoại lai hợp lệ | `id` không mang thông tin nguy cơ; ngoại lai y tế có thể là tín hiệu thật |
| Biến số | Median imputation + `StandardScaler` | Median bền vững với lệch/outlier; scaling cần cho Logistic Regression và SVM |
| Biến nhị phân | Mode imputation, giữ 0/1 | Không tạo giá trị giả giữa hai lớp |
| Biến phân loại | Mode imputation + One-Hot | Không áp đặt thứ tự giả; `handle_unknown='ignore'` giúp inference an toàn |
| Chia dữ liệu | 80/20 và `stratify=y`, `random_state=42` | Giữ tỷ lệ stroke ở hai tập và tái lập kết quả |
| Mất cân bằng | Xử lý ở model bằng `class_weight='balanced'` | Chỉ tác động khi train, không làm thay đổi test distribution |

**Đầu ra:** `preprocessor.joblib`, `preprocessed_split.joblib` và `models/schema.json`. Notebook 03 dùng split; notebook 04 dùng schema để đóng gói model.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from importlib import import_module
import json
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((root for root in SEARCH_ROOTS if (root / 'ai-models').is_dir()), Path.cwd())
WORK_DIR = Path('/content') if Path('/content').exists() else PROJECT_ROOT / '.colab_artifacts'
MODELS_DIR = WORK_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_NAME = 'healthcare-dataset-stroke-data.csv'
DATA_ROOTS = []
for root in SEARCH_ROOTS:
    DATA_ROOTS.extend([root / 'ai-models' / 'data', root / 'data'])
zip_candidates = [root / 'dataset.zip' for root in DATA_ROOTS]
zip_path = next((path for path in zip_candidates if path.exists()), None)

if zip_path is None:
    try:
        files = import_module('google.colab.files')
    except ImportError as error:
        raise FileNotFoundError('Khong tim thay dataset.zip. Dat file vao ai-models/data/ hoac upload tren Colab.') from error
    uploaded = files.upload()
    uploaded_path = Path('/content') / next(iter(uploaded))
    if uploaded_path.name != 'dataset.zip':
        raise ValueError('Vui long upload dung file dataset.zip.')
    zip_path = uploaded_path

with ZipFile(zip_path) as archive:
    csv_names = [name for name in archive.namelist() if Path(name).name == DATA_NAME]
    if not csv_names:
        raise FileNotFoundError(f'{DATA_NAME} khong co trong {zip_path}')
    with archive.open(csv_names[0]) as csv_file:
        df = pd.read_csv(csv_file)

print(f'ZIP input: {zip_path}')
print(f'CSV trong ZIP: {csv_names[0]}')
print('Shape ban dau:', df.shape)
print('Kieu du lieu:\n', df.dtypes)
print('Duplicate rows:', int(df.duplicated().sum()))
print('Missing values:\n', df.isna().sum()[df.isna().sum() > 0])

required_columns = {'id', 'stroke', 'age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease', 'gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Thieu cot bat buoc: {sorted(missing_columns)}')
if not set(df['stroke'].dropna().unique()).issubset({0, 1}):
    raise ValueError('Cot stroke phai chi gom 0 va 1.')

df = df.drop_duplicates().copy()
df = df.drop(columns=['id'])
NUMERIC_FEATURES = ['age', 'avg_glucose_level', 'bmi']
BINARY_FEATURES = ['hypertension', 'heart_disease']
CATEGORICAL_FEATURES = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
FEATURES = NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES
print('Shape sau lam sach:', df.shape)
print('Target rate:', f"{df['stroke'].mean():.2%}")


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
ZIP input: c:\Users\Admin\Desktop\17_12523101_10123337_DuDoanDotQuy\ai-models\data\dataset.zip
CSV trong ZIP: healthcare-dataset-stroke-data.csv
Shape ban dau: (5110, 12)
Kieu du lieu:
 id                     int64
gender                   str
age                  float64
hypertension           int64
heart_disease          int64
ever_married             str
work_type                str
Residence_type           str
avg_glucose_level    float64
bmi                  float64
smoking_status           str
stroke                 int64
dtype: object
Duplicate rows: 0
Missing values:
 bmi    200
dtype: int64
Shape sau lam sach: (5110, 11)
Target rate: 4.99%


In [3]:
X, y = df[FEATURES].copy(), df['stroke'].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Positive rate train: {y_train.mean():.2%} | test: {y_test.mean():.2%}')

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, NUMERIC_FEATURES),
    ('bin', binary_pipeline, BINARY_FEATURES),
    ('cat', categorical_pipeline, CATEGORICAL_FEATURES),
])

# Chỉ fit trên train; test chỉ được transform.
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)
print('Transformed train shape:', X_train_transformed.shape)
print('Transformed test shape:', X_test_transformed.shape)

Train: (4088, 10), Test: (1022, 10)
Positive rate train: 4.99% | test: 4.99%
Transformed train shape: (4088, 21)
Transformed test shape: (1022, 21)


In [4]:
joblib.dump({
    'X_train': X_train, 'X_test': X_test,
    'y_train': y_train, 'y_test': y_test,
    'feature_names': FEATURES,
}, WORK_DIR / 'preprocessed_split.joblib', compress=3)
joblib.dump(preprocessor, WORK_DIR / 'preprocessor.joblib', compress=3)

FEATURE_META = {
    'age': {'type': 'number', 'label': 'Tuoi', 'min': 0, 'max': 120},
    'avg_glucose_level': {'type': 'number', 'label': 'Muc duong huyet trung binh', 'min': 40, 'max': 300},
    'bmi': {'type': 'number', 'label': 'BMI', 'min': 10, 'max': 100},
    'hypertension': {'type': 'boolean', 'label': 'Cao huyet ap', 'values': [0, 1]},
    'heart_disease': {'type': 'boolean', 'label': 'Benh tim', 'values': [0, 1]},
    'gender': {'type': 'category', 'label': 'Gioi tinh', 'values': ['Male', 'Female', 'Other']},
    'ever_married': {'type': 'category', 'label': 'Da tung ket hon', 'values': ['Yes', 'No']},
    'work_type': {'type': 'category', 'label': 'Loai cong viec', 'values': ['children', 'Govt_job', 'Never_worked', 'Private', 'Self-employed']},
    'Residence_type': {'type': 'category', 'label': 'Noi sinh song', 'values': ['Rural', 'Urban']},
    'smoking_status': {'type': 'category', 'label': 'Tinh trang hut thuoc', 'values': ['formerly smoked', 'never smoked', 'smokes', 'Unknown']},
}
schema = {
    'target': 'stroke',
    'task': 'binary_classification',
    'features': [{'name': name, **meta} for name, meta in FEATURE_META.items()],
}
with open(MODELS_DIR / 'schema.json', 'w', encoding='utf-8') as file:
    json.dump(schema, file, ensure_ascii=False, indent=2)

print(f'Saved pipeline: {WORK_DIR / "preprocessor.joblib"}')
print(f'Saved split: {WORK_DIR / "preprocessed_split.joblib"}')
print(f'Saved schema: {MODELS_DIR / "schema.json"}')

Saved pipeline: \content\preprocessor.joblib
Saved split: \content\preprocessed_split.joblib
Saved schema: \content\models\schema.json
